[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S02_colecciones_control_flujo.ipynb)

# Sesión 02 · Colecciones y control de flujo

**Módulo 1: Python** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Guardar y manipular datos en listas (también en listas de listas) y tuplas.
2. Buscar, agregar y quitar datos en un diccionario.
3. Tomar decisiones con `if`, `elif` y `else` combinando comparadores con `and`, `or` y `not`.
4. Recorrer datos con `for` (con `range`, `enumerate` y `zip`) y con `while`, usando `break` y `continue`.

## 📋 Qué debes saber antes
Lo de la sesión 1: variables, tipos, operadores, índices y slicing en textos, y f-strings.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Si por error modificas los datos originales, vuelve a ejecutar el setup.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
from itertools import accumulate

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
ventas_semana = [int(v) for v in rng.integers(800, 5000, size=7)]
meta_diaria = 3000
umbral_bajo = 1500

catalogo_inicial = ["polo", "jean", "casaca", "gorra", "medias", "chompa"]

nombres_tiendas = ["Miraflores", "Surco", "Lince"]
ventas_tiendas = [[int(v) for v in rng.integers(5000, 20000, size=4)] for _ in range(3)]

registro_venta = ("2026-09-26", "Surco", "casaca", int(rng.integers(1, 6)), round(float(rng.uniform(80, 200)), 2))
caja_a = int(rng.integers(100, 500))
caja_b = int(rng.integers(500, 900))

precios_base = {p: round(float(rng.uniform(15, 150)), 2) for p in ["polo", "jean", "casaca", "gorra", "medias"]}

monto_compra = round(float(rng.uniform(50, 800)), 2)
es_frecuente = bool(rng.integers(0, 2))

ahorro_inicial = round(float(rng.uniform(100, 500)), 2)
aporte_semanal = round(float(rng.uniform(50, 150)), 2)
meta_ahorro = 2000

# ---------- Datos de práctica: movimientos bancarios ----------
saldo_inicial = round(float(rng.uniform(150, 400)), 2)
movimientos = []
for _d in range(1, 13):
    _fecha = f"2026-09-{_d:02d}"
    if _d == 6:
        movimientos.append((_fecha, "SUELDO", round(float(rng.uniform(2500, 3500)), 2)))
    elif _d == 9:
        movimientos.append((_fecha, "AJUSTE", 0.0))
    else:
        _c = str(rng.choice(["SUPERMERCADO", "RESTAURANTE", "CAJERO", "SERVICIOS"]))
        movimientos.append((_fecha, _c, -round(float(rng.uniform(20, 250)), 2)))

# Copia privada: los verificadores no dependen de lo que cambies arriba.
_D = copy.deepcopy({k: globals()[k] for k in [
    "dias", "ventas_semana", "meta_diaria", "umbral_bajo", "catalogo_inicial", "nombres_tiendas",
    "ventas_tiendas", "registro_venta", "caja_a", "caja_b", "precios_base", "monto_compra",
    "es_frecuente", "ahorro_inicial", "aporte_semanal", "meta_ahorro", "saldo_inicial", "movimientos",
]})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _sin_cambios(r, nombre):
    if globals().get(nombre) != _D[nombre]:
        r.mal(f"`{nombre}` cambió. No debes modificar los datos originales; vuelve a ejecutar el setup y revisa tu código.")


def check_ejercicio_1():
    r = _Revision("Ejercicio 1")
    v = _D["ventas_semana"]
    total = 0
    for x in v:
        total += x
    r.valor("venta_lunes", v[0], int, "revisa el índice del primer elemento")
    r.valor("venta_domingo", v[len(v) - 1], int, "revisa el índice del último elemento")
    r.valor("fin_de_semana", [v[5], v[6]], list, "deberían ser los dos últimos días, en su orden")
    r.valor("total_semana", total, int, "revisa la suma")
    r.valor("venta_max", sorted(v)[-1], int, "revisa el máximo")
    r.valor("promedio", round(total / 7, 2), float, "debería ser el total entre la cantidad de días, con 2 decimales",
            igual=lambda a, b: abs(a - b) <= 0.0051)
    r.valor("ordenadas_desc", sorted(v)[::-1], list, "deberían estar de mayor a menor")
    _sin_cambios(r, "ventas_semana")
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    base = _D["catalogo_inicial"]
    esperado = [p for p in base[:-1] if p != "gorra"] + ["bufanda"]
    r.valor("catalogo", esperado, list, "revisa los tres pasos y su orden")
    r.valor("retirado", base[-1], str, "debería ser el valor que devolvió `pop()`")
    r.valor("hay_jean", esperado.count("jean") > 0, bool, "debería ser el resultado de una pregunta con `in`")
    r.valor("n_catalogo", sum(1 for _ in esperado), int, "cuenta los elementos del catálogo final")
    _sin_cambios(r, "catalogo_inicial")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_sum_vacio": "25af31a57c2047d854d189042b0ecfb66843c4c19d3dd9ce1403e0be3826a240",
        "pred_in_vacio": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
        "pred_len_2d": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    vt = _D["ventas_tiendas"]
    r.valor("venta_surco_s3", vt[1][2], int, "revisa los dos índices: primero la fila (tienda), luego la columna (semana)")
    r.valor("ventas_lince", list(vt[2]), list, "debería ser la fila completa de Lince")
    r.valor("total_miraflores", vt[0][0] + vt[0][1] + vt[0][2] + vt[0][3], int, "suma la fila de Miraflores")
    r.valor("n_filas", 3, int, "cuenta las filas de la tabla")
    r.valor("n_columnas", 4, int, "cuenta los elementos de una fila")
    r.valor("ultima_semana", [fila[3] for fila in vt], list, "debería tener la semana 4 de cada tienda, en el orden de las filas")
    _sin_cambios(r, "ventas_tiendas")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    f, t, p, u, pr = _D["registro_venta"]
    for nombre, ref in (("fecha", f), ("tienda", t), ("producto", p), ("unidades", u), ("precio", pr)):
        r.valor(nombre, ref, type(ref), "revisa el orden de las variables al desempaquetar")
    r.valor("importe", round(u * pr, 2), float, "debería ser unidades por precio, con 2 decimales",
            igual=lambda a, b: abs(a - b) <= 0.0051)
    a = r.var("a")
    b = r.var("b")
    if a is not _FALTA and b is not _FALTA:
        if a == _D["caja_b"] and b == _D["caja_a"]:
            r.ok("`a` y `b` quedaron intercambiadas.")
        elif a == _D["caja_a"] and b == _D["caja_b"]:
            r.mal("`a` y `b` tienen los valores originales: falta el intercambio.")
        elif a == b:
            r.mal("`a` y `b` quedaron iguales: al intercambiar en dos líneas se pierde un valor. Hazlo en una sola línea.")
        else:
            r.mal("`a` y `b` no tienen los valores de `caja_a` y `caja_b` intercambiados.")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_tipo_1": "6b88ca3380698534b1380b457e554090cca21744b910883712503cb72f5e473a",
        "pred_tipo_2": "2ef0d86cc94a0960e013359fc270f1a50efdf896ddf7102af42fcbe173d4ed07",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5")
    pb = _D["precios_base"]
    esperado = {k: v for k, v in pb.items() if k != "gorra"}
    esperado["polo"] = round(pb["polo"] * 110 / 100, 2)
    esperado["bufanda"] = 49.9
    r.valor("precio_jean", pb["jean"], float, "debería ser el precio del jean en `precios`")
    r.valor("precio_bufanda", 0.0, float, "debería salir de `.get()` con `0.0` como valor por defecto")
    p = r.var("precios", dict)
    if p is not _FALTA:
        faltan = [k for k in esperado if k not in p]
        sobran = [k for k in p if k not in esperado]
        if faltan:
            r.mal(f"A `precios` le faltan estas claves: {faltan}.")
        if sobran:
            r.mal(f"En `precios` sobran estas claves: {sobran}.")
        if not faltan and not sobran:
            malos = [k for k in esperado if abs(p[k] - esperado[k]) > 0.0051]
            if malos:
                r.mal(f"Estos precios no son correctos: {malos}.")
            else:
                r.ok("`precios` tiene las claves y valores correctos.")
    r.valor("productos", list(esperado), list, "deberían ser las claves de `precios`, en su orden")
    r.valor("total_precios", round(math.fsum(esperado.values()), 2), float, "suma los valores de `precios` y redondea a 2 decimales",
            igual=lambda a, b: abs(a - b) <= 0.0051)
    _sin_cambios(r, "precios_base")
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6 · Parte A")
    m = _D["monto_compra"]
    esperado = next(pct for tope, pct in ((500, 15), (200, 10), (-math.inf, 0)) if m >= tope)
    if _D["es_frecuente"] and m >= 100:
        esperado += 5
    r.valor("descuento", esperado, int, "revisa el orden de las condiciones y la regla del cliente frecuente")
    r.valor("pago", round(m - m * esperado / 100, 2), float, "debería ser el monto menos el descuento, con 2 decimales",
            igual=lambda a, b: abs(a - b) <= 0.011)
    r.fin()
    r = _Revision("Ejercicio 6 · Parte B")
    r.predicciones({
        "pred_a": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
        "pred_b": "504fd320c035cf419fcfb0f0cbe7f3b478d10f78fda40022eeb35a86b23bce08",
        "pred_c": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
        "pred_d": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
    })
    r.fin()


def _etiqueta(x):
    meta = _D["meta_diaria"]
    return "alta" if x >= meta else ("media" if x * 2 >= meta else "baja")


def check_ejercicio_7():
    r = _Revision("Ejercicio 7")
    v = _D["ventas_semana"]
    r.valor("acumulado", list(accumulate(v)), list, "cada elemento debe ser la suma de todas las ventas hasta ese día")
    r.valor("variaciones", [b - a for a, b in zip(v, v[1:])], list,
            "debería tener 6 elementos: cada día menos el anterior, desde el martes")
    r.valor("etiquetas", [_etiqueta(x) for x in v], list, "revisa las condiciones y los límites (`>=`)")
    r.fin()


def check_ejercicio_8():
    r = _Revision("Ejercicio 8")
    v = _D["ventas_semana"]
    r.valor("pos_max", v.index(max(v)), int, "debería ser la posición (empezando en 0) del día con mayor venta")
    r.valor("dias_sobre_meta", [d for d, x in zip(_D["dias"], v) if x >= _D["meta_diaria"]], list,
            "deberían ser los nombres de los días que alcanzaron la meta, en orden")
    r.valor("productos_caros", [k for k, p in _D["precios_base"].items() if p > 50], list,
            "deberían ser los productos de `precios_base` que cuestan más de 50, en su orden")
    r.fin()


def check_ejercicio_9():
    r = _Revision("Ejercicio 9")
    faltante = _D["meta_ahorro"] - _D["ahorro_inicial"]
    semanas = max(0, math.ceil(faltante / _D["aporte_semanal"]))
    r.valor("semanas", semanas, int, "cuenta las vueltas del `while` hasta alcanzar la meta")
    r.valor("ahorro_final", round(_D["ahorro_inicial"] + semanas * _D["aporte_semanal"], 2), float,
            "debería ser el ahorro al salir del bucle, con 2 decimales", igual=lambda a, b: abs(a - b) <= 0.011)
    primero = next((d for d, x in zip(_D["dias"], _D["ventas_semana"]) if x < _D["umbral_bajo"]), None)
    r.valor("primer_dia_bajo", primero, str if primero is not None else type(None),
            "debería ser el primer día (en orden) con venta menor que `umbral_bajo`")
    r.valor("total_habiles", sum(_D["ventas_semana"][:5]), int, "suma solo de lunes a viernes")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    movs = _D["movimientos"]
    montos = [m[2] for m in movs]
    r.valor("saldo_final", round(_D["saldo_inicial"] + math.fsum(montos), 2), float,
            "debería ser el saldo inicial más todos los montos, con 2 decimales", igual=lambda a, b: abs(a - b) <= 0.011)
    r.valor("n_ingresos", len([x for x in montos if x > 0]), int, "cuenta solo los montos mayores que 0")
    r.valor("n_egresos", len([x for x in montos if x < 0]), int, "cuenta solo los montos menores que 0")
    gastos = {}
    for _, c, x in movs:
        if x < 0:
            gastos[c] = gastos.get(c, 0) - x
    gastos = {c: round(x, 2) for c, x in gastos.items()}
    g = r.var("gasto_por_concepto", dict)
    if g is not _FALTA:
        if set(g) != set(gastos):
            r.mal(f"`gasto_por_concepto` tiene las claves {sorted(g)}; deberían ser solo los conceptos con egresos.")
        elif any(v < 0 for v in g.values()):
            r.mal("En `gasto_por_concepto` hay montos negativos: guarda cuánto se gastó, en positivo.")
        elif all(abs(g[c] - gastos[c]) <= 0.011 for c in gastos):
            r.ok("`gasto_por_concepto` es correcto.")
        else:
            r.mal("`gasto_por_concepto` tiene las claves correctas, pero algún total no coincide.")
    r.valor("mayor_egreso", sorted(movs, key=lambda m: m[2])[0], tuple, "debería ser la tupla completa del movimiento más negativo")
    saldos = accumulate(montos, initial=_D["saldo_inicial"])
    next(saldos)
    r.valor("dias_sobregiro", [m[0] for m, s in zip(movs, saldos) if s < 0], list,
            "deberían ser las fechas en que el saldo, tras el movimiento, quedó por debajo de 0")
    _sin_cambios(r, "movimientos")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    r.valor("ventas_por_dia", {d: x for d, x in zip(_D["dias"], _D["ventas_semana"])}, dict,
            "cada día debería apuntar a su venta")
    mejor = {}
    for nombre, fila in zip(_D["nombres_tiendas"], _D["ventas_tiendas"]):
        mejor[nombre] = fila.index(max(fila)) + 1
    r.valor("mejor_semana", mejor, dict, "cada tienda debería apuntar al número de semana (1 a 4) con su mayor venta")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("dias             =", dias)
print("ventas_semana    =", ventas_semana)
print("meta_diaria      =", meta_diaria, "| umbral_bajo =", umbral_bajo)
print("catalogo_inicial =", catalogo_inicial)
print("nombres_tiendas  =", nombres_tiendas)
print("ventas_tiendas   =", ventas_tiendas)
print("registro_venta   =", registro_venta)
print("caja_a =", caja_a, "| caja_b =", caja_b)
print("precios_base     =", precios_base)
print("monto_compra     =", monto_compra, "| es_frecuente =", es_frecuente)
print("ahorro_inicial   =", ahorro_inicial, "| aporte_semanal =", aporte_semanal, "| meta_ahorro =", meta_ahorro)
print()
print("🏦 Movimientos bancarios")
print("saldo_inicial =", saldo_inicial)
for m in movimientos:
    print("  ", m)

---
## 1. Listas: acceder y resumir

### 📘 Concepto
Una **lista** guarda varios valores en orden, entre corchetes: `[10, 20, 30]`. Se indexa igual que un texto: el primero es `[0]`, el último `[-1]`, y `[inicio:fin]` devuelve otra lista.

| Función | Qué devuelve |
|---|---|
| `len(lista)` | cantidad de elementos |
| `sum(lista)` | suma (solo números) |
| `max(lista)` / `min(lista)` | mayor / menor |
| `sorted(lista)` | una lista **nueva** ordenada de menor a mayor; con `reverse=True`, de mayor a menor |

`sorted()` no toca la lista original. En cambio, `lista.sort()` la ordena **en su sitio** y la cambia para siempre.

In [ ]:
unidades_ej = [4, 1, 7, 3]
print(unidades_ej[0], unidades_ej[-1], unidades_ej[1:3])
print(len(unidades_ej), sum(unidades_ej), max(unidades_ej), min(unidades_ej))
print(sorted(unidades_ej))
print(sorted(unidades_ej, reverse=True))
print(unidades_ej)            # la original sigue igual

### ✍️ Tu turno · Ejercicio 1: la semana en números
`ventas_semana` tiene las ventas de lunes a domingo. Crea:
1. `venta_lunes` y `venta_domingo` (para el domingo usa un índice negativo).
2. `fin_de_semana`: una lista con las ventas de sábado y domingo, usando slicing.
3. `total_semana` y `venta_max`.
4. `promedio`: venta promedio por día, redondeada a 2 decimales.
5. `ordenadas_desc`: las ventas de mayor a menor, **sin modificar** `ventas_semana`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Para el promedio divide el total entre la cantidad de elementos (usa `len`, no escribas el 7 a mano).
</details>

<details><summary>💡 Pista 2</summary>

Para `ordenadas_desc` usa la función que devuelve una lista nueva y pásale el parámetro que invierte el orden. Si `ventas_semana` quedó modificada, vuelve a ejecutar el setup.
</details>

---
## 2. Listas: modificar y buscar

### 📘 Concepto
Las listas se pueden cambiar:

| Método | Qué hace |
|---|---|
| `lista.append(x)` | agrega `x` al final |
| `lista.remove(x)` | quita la primera aparición de `x` (error si no está) |
| `lista.pop()` | quita el último y lo **devuelve**; `pop(i)` quita el de la posición `i` |
| `x in lista` | `True` si `x` está en la lista |

⚠️ `copia = lista` **no copia**: las dos variables apuntan a la misma lista y cambiar una cambia la otra. Para una copia de verdad usa `lista.copy()`.

In [ ]:
pedido = ["polo", "jean"]
alias = pedido            # el mismo objeto con otro nombre
copia = pedido.copy()     # una lista independiente

pedido.append("gorra")
print(pedido, alias, copia)

ultimo = pedido.pop()
print("Saqué:", ultimo, "| queda:", pedido)
print("jean" in pedido, "zapato" in pedido)

### ✍️ Tu turno · Ejercicio 2: actualizar el catálogo
**Parte A.**
1. Crea `catalogo` como **copia** de `catalogo_inicial`.
2. Saca el último producto de `catalogo` con `pop()` y guárdalo en `retirado`.
3. Agrega `"bufanda"` al final.
4. Quita `"gorra"`.
5. Crea `hay_jean` (`True` o `False`, usando `in`) y `n_catalogo` (cantidad de productos).

`catalogo_inicial` no debe cambiar.

**Parte B · casos borde.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_sum_vacio` | ¿qué valor da `sum([])`? |
| `pred_in_vacio` | ¿qué valor da `"polo" in []`? |
| `pred_len_2d` | ¿qué valor da `len([[1, 2, 3], [4, 5, 6]])`? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Empieza con `catalogo = catalogo_inicial.copy()`. Así, si vuelves a ejecutar la celda, parte siempre del catálogo original.
</details>

<details><summary>💡 Pista 2</summary>

`pop()` devuelve el elemento que quita: guárdalo con `retirado = catalogo.pop()`. En la parte B, piensa qué cuenta `len` en una lista cuyos elementos son listas.
</details>

---
## 3. Listas de listas (2D)

### 📘 Concepto
Una lista puede contener listas: así se guarda una tabla. `tabla[i]` es la fila `i` y `tabla[i][j]` es el valor de la fila `i`, columna `j`.

In [ ]:
# filas: 2 vendedores; columnas: 3 días
tabla_ej = [
    [120, 340, 90],
    [200, 150, 310],
]
print(tabla_ej[1])        # fila del segundo vendedor
print(tabla_ej[1][2])     # segundo vendedor, tercer día
print(len(tabla_ej), len(tabla_ej[0]))   # filas y columnas

### ✍️ Tu turno · Ejercicio 3: ventas por tienda y semana
En `ventas_tiendas` cada fila es una tienda (en el orden de `nombres_tiendas`) y cada columna una semana (de la 1 a la 4). Crea:
1. `venta_surco_s3`: la venta de Surco en la semana 3.
2. `ventas_lince`: la fila completa de Lince.
3. `total_miraflores`: la venta total de Miraflores en las 4 semanas.
4. `n_filas` y `n_columnas`, usando `len`.
5. `ultima_semana`: una lista con la venta de la semana 4 de cada tienda, en el orden de las filas.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Los índices empiezan en 0: la semana 3 es la columna 2.
</details>

<details><summary>💡 Pista 2</summary>

Para `ultima_semana` arma una lista nueva con tres elementos, y en cada uno toma el último valor de una fila.
</details>

---
## 4. Tuplas y desempaquetado

### 📘 Concepto
Una **tupla** es como una lista pero **no se puede modificar**. Se escribe con paréntesis: `("Surco", 3, 39.9)`. Sirve para registros fijos, como una fila de datos.

**Desempaquetar** es asignar cada elemento a una variable en una sola línea: `a, b, c = tupla`. Tiene que haber tantas variables como elementos. Así también se intercambian dos variables: `x, y = y, x`.

Lo que define una tupla es la coma, no los paréntesis.

In [ ]:
venta_ej = ("2026-09-26", "Lince", 2)
dia_ej, lugar_ej, cantidad_ej = venta_ej
print(dia_ej, lugar_ej, cantidad_ej)

x, y = 1, 2
x, y = y, x
print(x, y)

# venta_ej[2] = 5   # TypeError: una tupla no se puede modificar

### ✍️ Tu turno · Ejercicio 4: desempaquetar e intercambiar
**Parte A.**
1. Desempaqueta `registro_venta` en `fecha`, `tienda`, `producto`, `unidades` y `precio`, en una sola línea.
2. Crea `importe` = unidades por precio, redondeado a 2 decimales.
3. Crea `a = caja_a` y `b = caja_b`. Luego intercambia `a` y `b` en **una sola línea**.

**Parte B.** Predice **sin ejecutar** y escribe el nombre del tipo como texto (por ejemplo, `"list"`):

| Variable | Pregunta |
|---|---|
| `pred_tipo_1` | ¿de qué tipo es `(5,)`? |
| `pred_tipo_2` | ¿de qué tipo es `(5)`? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

El orden de las variables al desempaquetar tiene que ser el mismo que el de la tupla. Imprime `registro_venta` para verlo.
</details>

<details><summary>💡 Pista 2</summary>

Si intercambias en dos líneas (`a = b` y luego `b = a`) pierdes un valor. Usa la forma `x, y = y, x`. En la parte B, recuerda qué define a una tupla.
</details>

---
## 5. Diccionarios

### 📘 Concepto
Un **diccionario** guarda pares `clave: valor`. Se busca por clave, no por posición.

| Operación | Qué hace |
|---|---|
| `d["polo"]` | devuelve el valor de esa clave (`KeyError` si no existe) |
| `d.get("polo", 0)` | igual, pero si la clave no existe devuelve `0` (o `None` si no das un valor) |
| `d["gorra"] = 25` | agrega la clave o cambia su valor |
| `del d["gorra"]` | borra la clave |
| `d.keys()`, `d.values()`, `d.items()` | claves, valores y pares `(clave, valor)` |

Para convertir `keys()` o `values()` en una lista usa `list(...)`. Igual que con las listas, `d.copy()` crea una copia independiente.

In [ ]:
stock_ej = {"polo": 12, "jean": 5}
print(stock_ej["polo"])
print(stock_ej.get("casaca"), stock_ej.get("casaca", 0))

stock_ej["casaca"] = 8
stock_ej["polo"] = stock_ej["polo"] - 2
del stock_ej["jean"]
print(stock_ej)
print(list(stock_ej.keys()), list(stock_ej.values()), sum(stock_ej.values()))
print(list(stock_ej.items()))

### ✍️ Tu turno · Ejercicio 5: lista de precios
1. Crea `precios` como copia de `precios_base`.
2. `precio_jean`: el precio del jean.
3. `precio_bufanda`: el precio de la bufanda usando `.get()` con `0.0` como valor por defecto (todavía no existe).
4. Agrega la bufanda a `precios` a 49.9.
5. Borra la gorra de `precios`.
6. Sube el precio del polo en `precios` un 10 %, redondeado a 2 decimales.
7. `productos`: lista con las claves de `precios`.
8. `total_precios`: suma de los precios de `precios`, redondeada a 2 decimales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Sigue el orden de los pasos. `precio_bufanda` se calcula **antes** de agregar la bufanda.
</details>

<details><summary>💡 Pista 2</summary>

Para subir el polo, lee su valor actual, calcula el nuevo y guárdalo en la misma clave. `sum()` funciona sobre `precios.values()`.
</details>

---
## 6. Decisiones: `if`, `elif`, `else`

### 📘 Concepto
**Comparadores:** `==` (igual), `!=` (distinto), `<`, `<=`, `>`, `>=`. Devuelven `True` o `False`. Se pueden encadenar: `0 < x < 10`.

**Lógicos:** `and` (las dos), `or` (al menos una), `not` (invierte). `not` se aplica antes que `and`, y `and` antes que `or`.

`if` ejecuta un bloque solo si la condición es verdadera. El bloque se marca con `:` y **sangría** (4 espacios). Python revisa las condiciones de arriba abajo y ejecuta **solo la primera** que se cumple.

```python
if condicion_1:
    ...
elif condicion_2:
    ...
else:
    ...
```

Un valor usado como condición se convierte con `bool()`: `0`, `""` y las colecciones vacías cuentan como `False`.

In [ ]:
stock_ej = 3
es_temporada = True

if stock_ej == 0:
    estado = "agotado"
elif stock_ej < 5 and es_temporada:
    estado = "reponer urgente"
elif stock_ej < 5:
    estado = "reponer"
else:
    estado = "ok"
print(estado)

print(5 != 3, 2 <= 2 < 1, not es_temporada)

### ✍️ Tu turno · Ejercicio 6: descuento por compra
**Parte A.** Calcula `descuento` (un `int`, en porcentaje) para `monto_compra` con estas reglas:
- 500 o más: 15.
- Desde 200 hasta menos de 500: 10.
- Menos de 200: 0.
- Además, si `es_frecuente` es verdadero **y** el monto es de 100 o más, suma 5 puntos más.

Luego calcula `pago`: el monto con el descuento aplicado, redondeado a 2 decimales.

**Parte B.** Predice **sin ejecutar** (`True` o `False`):

| Variable | Expresión |
|---|---|
| `pred_a` | `10 >= 10 and 3 != 3` |
| `pred_b` | `not 5 > 2 or 4 == 4` |
| `pred_c` | `bool([])` |
| `pred_d` | `1 < 5 < 3` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Ordena las condiciones de la más exigente a la menos exigente: como solo se ejecuta la primera que se cumple, el orden importa.
</details>

<details><summary>💡 Pista 2</summary>

La regla del cliente frecuente es un `if` aparte, después del bloque `if`/`elif`/`else`, y usa `descuento += 5`. En `pred_d`, una comparación encadenada equivale a unir las dos con `and`.
</details>

---
## 7. Bucles `for`: recorrer listas y `range`

### 📘 Concepto
`for` repite un bloque una vez por cada elemento:

```python
for elemento in lista:
    ...
```

`range` genera números: `range(5)` da 0, 1, 2, 3, 4; `range(2, 5)` da 2, 3, 4; `range(0, 10, 3)` da 0, 3, 6, 9. Sirve para repetir algo *n* veces o para recorrer posiciones.

Patrón clave del análisis de datos: **acumular**. Antes del bucle creas un total o una lista vacía, y dentro del bucle los vas completando.

In [ ]:
unidades_ej = [4, 1, 7, 3]

total_ej = 0
dobles_ej = []
for u in unidades_ej:
    total_ej += u
    dobles_ej.append(u * 2)
print(total_ej, dobles_ej)

for i in range(1, len(unidades_ej)):
    print("posición", i, "valor", unidades_ej[i], "anterior", unidades_ej[i - 1])

### ✍️ Tu turno · Ejercicio 7: recorrer la semana
Usa bucles `for` sobre `ventas_semana`:
1. `acumulado`: lista en la que cada elemento es el total vendido **desde el lunes hasta ese día**.
2. `variaciones`: lista con la diferencia de cada día respecto del día anterior, desde el martes (tendrá 6 elementos). Usa `range`.
3. `etiquetas`: lista con `"alta"` si la venta alcanza `meta_diaria`, `"media"` si alcanza la mitad de la meta y `"baja"` en otro caso.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_7()

<details><summary>💡 Pista 1</summary>

Cada lista empieza vacía (`[]`) antes de su bucle y se llena con `append`. Para `acumulado` lleva además un total que vas sumando.
</details>

<details><summary>💡 Pista 2</summary>

Para `variaciones`, recorre las posiciones con `range(1, len(ventas_semana))` y resta el valor de la posición anterior (`i - 1`). Para `etiquetas`, pon un `if`/`elif`/`else` dentro del bucle.
</details>

---
## 8. `enumerate`, `zip` e `.items()`

### 📘 Concepto
- `enumerate(lista)` da pares `(posición, valor)`: `for i, x in enumerate(lista)`.
- `zip(a, b)` recorre dos listas en paralelo: `for x, y in zip(a, b)`. Se detiene en la más corta.
- `d.items()` da pares `(clave, valor)` de un diccionario: `for k, v in d.items()`.

En los tres casos se desempaqueta en el propio `for`.

In [ ]:
productos_ej = ["polo", "jean", "gorra"]
stock_ej = [12, 0, 7]

for i, p in enumerate(productos_ej):
    print(i, p)

for p, s in zip(productos_ej, stock_ej):
    if s == 0:
        print("Sin stock:", p)

for producto, precio in {"polo": 39.9, "jean": 99.9}.items():
    print(f"{producto}: S/ {precio:.2f}")

### ✍️ Tu turno · Ejercicio 8: posiciones y pares
1. `pos_max`: la posición (empezando en 0) del día con mayor venta en `ventas_semana`. Usa `enumerate`; si hay empate, quédate con la primera.
2. `dias_sobre_meta`: lista con los **nombres** de los días (de `dias`) cuya venta alcanza `meta_diaria`. Usa `zip`.
3. `productos_caros`: lista con los productos de `precios_base` que cuestan más de 50. Usa `.items()`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_8()

<details><summary>💡 Pista 1</summary>

Para `pos_max` necesitas recordar dos cosas mientras recorres: la mejor venta vista hasta ahora y su posición.
</details>

<details><summary>💡 Pista 2</summary>

Empieza con `pos_max = 0` y compara cada venta con `ventas_semana[pos_max]`. Usa `>` (no `>=`) para quedarte con la primera en caso de empate.
</details>

---
## 9. `while`, `break` y `continue`

### 📘 Concepto
- `while condicion:` repite **mientras** la condición sea verdadera. Útil cuando no sabes cuántas vueltas harán falta. Asegúrate de que algo cambie dentro del bucle o no terminará nunca (si pasa, detén la celda con ⏹️).
- `break` sale del bucle en ese momento.
- `continue` salta al siguiente elemento sin ejecutar el resto del bloque.

In [ ]:
deuda_ej = 1000
meses_ej = 0
while deuda_ej > 0:
    deuda_ej -= 300
    meses_ej += 1
print("Meses para pagar:", meses_ej, "| saldo final:", deuda_ej)

for p in ["polo", "jean", "gorra", "casaca"]:
    if p == "jean":
        continue          # salta el jean
    if p == "casaca":
        break             # termina antes de llegar al final
    print(p)

### ✍️ Tu turno · Ejercicio 9: ahorro y búsquedas
1. Con `while`: partiendo de `ahorro_inicial` y sumando `aporte_semanal` cada semana, ¿cuántas semanas hacen falta para llegar al menos a `meta_ahorro`? Guarda `semanas` y `ahorro_final` (redondeado a 2 decimales).
2. Con `break`: `primer_dia_bajo` es el nombre del primer día cuya venta es menor que `umbral_bajo`. Empieza con `primer_dia_bajo = None` para cubrir el caso en que ningún día cumple.
3. Con `continue`: `total_habiles` es la suma de las ventas **saltando** sábado y domingo.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_9()

<details><summary>💡 Pista 1</summary>

En el `while`, la condición es "todavía no llego a la meta". Dentro, suma el aporte y cuenta una semana.
</details>

<details><summary>💡 Pista 2</summary>

Para los puntos 2 y 3 recorre `zip(dias, ventas_semana)`. En el 2, cuando encuentres el día, guárdalo y usa `break`. En el 3, si el día es `"sábado"` o `"domingo"`, usa `continue` antes de sumar.
</details>

---
## 🏋️ Reto final: el extracto del mes
`movimientos` es una lista de tuplas `(fecha, concepto, monto)`: los montos positivos son ingresos y los negativos, egresos. `saldo_inicial` es el saldo antes del primer movimiento. Hay un movimiento con monto 0 (un ajuste): no es ingreso ni egreso.

Crea:
1. `saldo_final`: redondeado a 2 decimales.
2. `n_ingresos` y `n_egresos`.
3. `gasto_por_concepto`: diccionario `concepto → total gastado`, solo con egresos y en **positivo**, cada total redondeado a 2 decimales.
4. `mayor_egreso`: la tupla completa del movimiento con el monto más negativo.
5. `dias_sobregiro`: lista con las fechas en que el saldo, **después** del movimiento, quedó por debajo de 0.

No modifiques `movimientos`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Un solo bucle `for fecha, concepto, monto in movimientos:` puede resolver casi todo: ve actualizando un saldo, los contadores y el diccionario.
</details>

<details><summary>💡 Pista 2</summary>

Para `gasto_por_concepto` usa `d[c] = d.get(c, 0) + ...` para sumar aunque la clave aún no exista; redondea al final, en un segundo bucle sobre `.items()`. Para `mayor_egreso`, guarda la tupla completa cada vez que encuentres un monto menor que el mínimo visto.
</details>

---
## 🚀 Nivel pro (opcional)
1. `ventas_por_dia`: un diccionario `día → venta` construido en **una línea** combinando `dict()` y `zip()`.
2. `mejor_semana`: un diccionario `tienda → número de semana (1 a 4)` en el que cada tienda tuvo su mayor venta, usando `nombres_tiendas` y `ventas_tiendas`. Pista: un bucle dentro de otro, o investiga el método `.index()` de las listas.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar la diferencia entre `sorted(lista)` y `lista.sort()`.
- [ ] Explicar por qué `b = a` no copia una lista y cómo hacer una copia.
- [ ] Acceder a un valor de una lista de listas con dos índices.
- [ ] Desempaquetar una tupla e intercambiar dos variables en una línea.
- [ ] Explicar la diferencia entre `d["clave"]` y `d.get("clave", 0)`.
- [ ] Escribir un `if`/`elif`/`else` y saber por qué importa el orden de las condiciones.
- [ ] Acumular un total o una lista dentro de un `for`.
- [ ] Usar `enumerate`, `zip` e `.items()` para recorrer dos cosas a la vez.
- [ ] Elegir entre `for` y `while`, y usar `break` y `continue`.

**Próxima sesión (S03):** funciones, casos borde, módulos y comprensiones de listas.